# ALSApp - Plant disease classifier (on-device)

Trains a small MobileNetV2-based classifier on the public **PlantVillage** dataset
(38 classes across 14 crop species, ~54k leaf photos) and exports it as a `.tflite`
file small enough to ship inside the Flutter app and run fully offline on the phone.

**Before running:** `Runtime` -> `Change runtime type` -> select a `T4 GPU`. No dataset
download/auth needed - PlantVillage comes bundled in TensorFlow Datasets.

**At the end** this notebook downloads two files - save them into
`frontend/assets/models/` in the app repo:
- `plant_disease.tflite`
- `plant_labels.txt`

In [ ]:
!pip install -q -U tensorflow tensorflow-datasets scikit-learn

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import classification_report

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

# plant_village only ships a single 'train' split - carve out a held-out
# slice ourselves so we have an honest accuracy number at the end.
(train_raw, val_raw), ds_info = tfds.load(
    'plant_village',
    split=['train[:90%]', 'train[90%:]'],
    as_supervised=True,
    with_info=True,
)

class_names = ds_info.features['label'].names
num_classes = ds_info.features['label'].num_classes
print(f"{num_classes} classes, e.g.: {class_names[:5]}")

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

def prep(image, label, training):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    if training:
        image = augment(image, training=True)
    return image, label

train_ds = (train_raw
            .map(lambda x, y: prep(x, y, True), num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(2000)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))
val_ds = (val_raw
          .map(lambda x, y: prep(x, y, False), num_parallel_calls=tf.data.AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))

In [ ]:
# Rescaling is baked into the model itself (not done in the tf.data pipeline)
# so the Flutter side can feed plain 0-255 RGB pixels with no normalization math.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.layers.Rescaling(1. / 127.5, offset=-1)(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Phase 1: train just the new head, backbone frozen.
model.fit(train_ds, validation_data=val_ds, epochs=6)

In [ ]:
# Phase 2: unfreeze the top of the backbone and fine-tune a few epochs at a
# low learning rate - usually a solid accuracy bump for not much extra time.
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=4)

In [ ]:
# Honest held-out numbers before we ship this anywhere.
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# NOTE: no converter.optimizations (dynamic-range quantization) on purpose. TF >= 2.13 then emits
# FULLY_CONNECTED v12, which the TFLite runtimes bundled by tflite_flutter on macOS (2.11) and
# iOS (2.12) cannot load ("Didn't find op for builtin opcode FULLY_CONNECTED version 12").
# Plain float32 is ~9 MB and runs on every platform.
tflite_model = converter.convert()

with open('plant_disease.tflite', 'wb') as f:
    f.write(tflite_model)
with open('plant_labels.txt', 'w') as f:
    f.write('\n'.join(class_names))

import os
print(f"plant_disease.tflite: {os.path.getsize('plant_disease.tflite') / 1e6:.1f} MB")

In [ ]:
from google.colab import files
files.download('plant_disease.tflite')
files.download('plant_labels.txt')